In [1]:
from langfuse import get_client
import dotenv 
dotenv.load_dotenv()  # Load environment variables from .env file

langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

from pydantic_ai.agent import Agent

# Initialize Pydantic AI instrumentation
Agent.instrument_all()

Langfuse client is authenticated and ready!


In [2]:
from pydantic_ai import Agent, RunContext

roulette_agent = Agent(
    'openrouter:anthropic/claude-sonnet-4-6',
    deps_type=int,
    system_prompt=(
        'Use the `roulette_wheel` function to see if the '
        'customer has won based on the number they provide.'
    ),
    instrument=True
)

@roulette_agent.tool
async def roulette_wheel(ctx: RunContext[int], square: int) -> str:
    """check if the square is a winner"""
    return 'winner' if square == ctx.deps else 'loser'

Demo of observe in langfuse.

In [3]:
from langfuse import observe

@observe(name="roulette_round")
async def play_round(user_guess: int, winning_number: int) -> str:
    result = await roulette_agent.run(
        f"My number is {user_guess}. Did I win?",
        deps=winning_number,
    )
    return result.output

@observe(name="roulette_session")
async def play_session(rounds: list[tuple[int, int]]) -> list[str]:
    outputs: list[str] = []
    for guess, winner in rounds:
        outcome = await play_round(user_guess=guess, winning_number=winner)
        line = f"guess={guess}, winner={winner} -> {outcome}"
        print(line)
        outputs.append(line)
    return outputs

# A single observed call creates one trace with nested operations.
rounds = [
    (7, 7),   # winner
    (13, 7),  # loser
    (22, 22), # winner
]

_ = await play_session(rounds)
print("Open Langfuse and filter by trace/operation name: roulette_session")

guess=7, winner=7 -> 🎉 Congratulations, you're a **winner**! 🎉 Your number, **7**, is the lucky number on the roulette wheel! Well done! 🍀
guess=13, winner=7 -> Unfortunately, it looks like **13** is not a winning number this time. Better luck next time! 🍀 Would you like to try another number?
guess=22, winner=22 -> 🎉 Congratulations, you're a **winner**! 🎉 The roulette wheel landed on **22** and that's a winning square! Well done! 🥳
Open Langfuse and filter by trace/operation name: roulette_session


## Structured Output

pydantic-ai agents can return typed Pydantic models instead of free text. The LLM's response is automatically validated against the schema — if it doesn't conform, pydantic-ai retries. This is how IslandSim gets reliable, parseable game state from every agent call.

In [8]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
import time

class CityProfile(BaseModel):
    name: str = Field(description="City name")
    country: str = Field(description="Country the city is in")
    population_millions: float = Field(description="Approximate population in millions")
    famous_for: list[str] = Field(description="Top 3 things the city is known for")
    walkability_score: int = Field(ge=1, le=10, description="Walkability from 1-10")

# output_type tells pydantic-ai to parse the LLM output into this model
city_agent = Agent(
    'openrouter:anthropic/claude-sonnet-4-6',
    output_type=CityProfile,
    system_prompt='You are a travel guide. Return accurate city profiles.',
    instrument=True
)

start = time.time()
result = await city_agent.run('Tell me about Tokyo.')
elapsed = time.time() - start
profile = result.output

# The output is a fully typed Pydantic model, not a string
print(f"City:          {profile.name}")
print(f"Country:       {profile.country}")
print(f"Population:    {profile.population_millions}M")
print(f"Famous for:    {', '.join(profile.famous_for)}")
print(f"Walkability:   {profile.walkability_score}/10")
print(f"\nType: {type(profile).__name__}")
print(f"Raw dict: {profile.model_dump()}")

# Diagnostics: timing, retries, and token usage
messages = result.all_messages()
print(f"\n--- Diagnostics ---")
print(f"Elapsed:         {elapsed:.1f}s")
print(f"Total messages:  {len(messages)}  (3 = clean, more = retries)")
print(f"Usage:           {result.usage()}")

# Show the underlying messages — pydantic-ai uses a tool call to extract structured output
print("\n--- Message History ---")
for msg in messages:
    print(f"\n[{msg.kind}]")
    if hasattr(msg, 'parts'):
        for part in msg.parts:
            print(f"  {part}")

City:          Tokyo
Country:       Japan
Population:    13.96M
Famous for:    World-class cuisine and Michelin-starred restaurants, Cutting-edge technology and pop culture (anime, manga, gaming), Historic temples, shrines, and imperial palaces
Walkability:   8/10

Type: CityProfile
Raw dict: {'name': 'Tokyo', 'country': 'Japan', 'population_millions': 13.96, 'famous_for': ['World-class cuisine and Michelin-starred restaurants', 'Cutting-edge technology and pop culture (anime, manga, gaming)', 'Historic temples, shrines, and imperial palaces'], 'walkability_score': 8}

--- Diagnostics ---
Elapsed:         4.0s
Total messages:  3  (3 = clean, more = retries)
Usage:           RunUsage(input_tokens=818, output_tokens=155, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)

--- Message History ---

[request]
  SystemPromptPart(content='You are a travel guide. Return accurate city profiles.', timestamp=datetime.datetime(2026, 3, 24, 19, 45, 50, 